In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/appledisease/Test/Blotch_Apple/149.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/143.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/131.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/130.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/120.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/124.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/123.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/137.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/127.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/140.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/145.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/135.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/121.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/126.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/142.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/148.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/138.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/146.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/1

In [1]:
!pip install protobuf==3.20.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 3.4 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tens

In [2]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import InceptionV3, EfficientNetB3, MobileNetV2
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense, Input
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.optimizers import Adam
import os
import numpy as np
from tensorflow.keras.models import load_model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

2025-11-28 01:36:34.100629: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764293794.364585      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764293794.443119      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
img_size = (224, 224)
batch_size = 32
train_dir = "/kaggle/input/appledisease/Train"
test_dir = "/kaggle/input/appledisease/Test"

In [4]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2
)

In [5]:
test_datagen = ImageDataGenerator(rescale=1./255)

In [6]:
train_gen = train_datagen.flow_from_directory(
    train_dir, target_size=img_size, batch_size=batch_size, 
    class_mode='categorical', subset='training'
)

Found 307 images belonging to 4 classes.


In [7]:
val_gen = train_datagen.flow_from_directory(
    train_dir, target_size=img_size, batch_size=batch_size, 
    class_mode='categorical', subset='validation'
)

Found 75 images belonging to 4 classes.


In [8]:
test_gen = test_datagen.flow_from_directory(
    test_dir, target_size=img_size, batch_size=batch_size, 
    class_mode='categorical', shuffle=False 
)

Found 120 images belonging to 4 classes.


In [9]:
def build_model(model_name, num_classes):
    inputs = Input(shape=(224, 224, 3))
    
    if model_name == 'ResNet50':
        base_model = ResNet50(include_top=False, weights='imagenet', input_tensor=inputs)
    elif model_name == 'DenseNet121':
        base_model = DenseNet121(include_top=False, weights='imagenet', input_tensor=inputs)
    elif model_name == 'EfficientNetB0':
        base_model = EfficientNetB0(include_top=False, weights='imagenet', input_tensor=inputs)
    base_model.trainable = False
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dropout(0.3)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs, name=model_name)
    
    model.compile(optimizer=Adam(learning_rate=1e-3), 
                  loss='categorical_crossentropy', 
                  metrics=['accuracy'])
    return model

In [10]:
model_names = ['ResNet50', 'DenseNet121', 'EfficientNetB0']
trained_models = {}

In [11]:
for name in model_names:
    print(f"\n{'='*40}")
    print(f" TRAINING MODEL: {name}")
    print(f"{'='*40}")
    model = build_model(name, train_gen.num_classes)
    
    # Callbacks
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)
    
    # Train
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=10,  # Increase this if you have time
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )
    
    # Save the model
    save_path = f"{name}_apple_final.h5"
    model.save(save_path)
    print(f"✅ Saved {name} to {save_path}")
    
    # Clear memory to prevent crashing
    tf.keras.backend.clear_session()


 TRAINING MODEL: ResNet50


2025-11-28 01:37:11.026163: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 54s 4s/step - accuracy: 0.2704 - loss: 1.6612 - val_accuracy: 0.3067 - val_loss: 1.3782 - learning_rate: 0.0010
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 36s 4s/step - accuracy: 0.2598 - loss: 1.4736 - val_accuracy: 0.3200 - val_loss: 1.3531 - learning_rate: 0.0010
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 36s 4s/step - accuracy: 0.3107 - loss: 1.4379 - val_accuracy: 0.2933 - val_loss: 1.3428 - learning_rate: 0.0010
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 36s 4s/step - accuracy: 0.2892 - loss: 1.4914 - val_accuracy: 0.3600 - val_loss: 1.3323 - learning_rate: 0.0010
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 36s 4s/step - accuracy: 0.2938 - loss: 1.3926 - val_accuracy: 0.3600 - val_loss: 1.3431 - learning_rate: 0.0010
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 36s 4s/step - accuracy: 0.2993 - loss: 1.3665 - val_accuracy: 0.3600 - val_loss: 1.3359 - learning_rate: 0.0010
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 36s 4s/step - accuracy: 0.3258 - loss: 1.3615 - val_accuracy: 

✅ Saved ResNet50 to ResNet50_apple_final.h5

 TRAINING MODEL: DenseNet121
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 73s 5s/step - accuracy: 0.3153 - loss: 1.9818 - val_accuracy: 0.5467 - val_loss: 1.0484 - learning_rate: 0.0010
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 38s 4s/step - accuracy: 0.5581 - loss: 1.1576 - val_accuracy: 0.8000 - val_loss: 0.7164 - learning_rate: 0.0010
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 38s 4s/step - accuracy: 0.7349 - loss: 0.7250 - val_accuracy: 0.8000 - val_loss: 0.5768 - learning_rate: 0.0010
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 38s 4s/step - accuracy: 0.7838 - loss: 0.6590 - val_accuracy: 0.8800 - val_loss: 0.4837 - learning_rate: 0.0010
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 38s 4s/step - accuracy: 0.8215 - loss: 0.5249 - val_accuracy: 0.8533 - val_loss: 0.4119 - learning_rate: 0.0010
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 38s 4s/step - accuracy: 0.8067 - loss: 0.5032 - val_accuracy: 0.8267 - val_loss: 0.3797

✅ Saved DenseNet121 to DenseNet121_apple_final.h5

 TRAINING MODEL: EfficientNetB0
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 44s 3s/step - accuracy: 0.2662 - loss: 1.4362 - val_accuracy: 0.2933 - val_loss: 1.4161 - learning_rate: 0.0010
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 19s 2s/step - accuracy: 0.2872 - loss: 1.4250 - val_accuracy: 0.3067 - val_loss: 1.3724 - learning_rate: 0.0010
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 19s 2s/step - accuracy: 0.2817 - loss: 1.4022 - val_accuracy: 0.2933 - val_loss: 1.3779 - learning_rate: 0.0010
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 19s 2s/step - accuracy: 0.3088 - loss: 1.4005 - val_accuracy: 0.3067 - val_loss: 1.3640 - learning_rate: 0.0010
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 19s 2s/step - accuracy: 0.3330 - loss: 1.3730 - val_accuracy: 0.3067 - val_loss: 1.3625 - learning_rate: 0.0010
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 19s 2s/step - accuracy: 0.2697 - loss: 1.3658 - val_accuracy: 0.2933 - val_los

✅ Saved EfficientNetB0 to EfficientNetB0_apple_final.h5


In [12]:
model_res = load_model('ResNet50_apple_final.h5')
model_dense = load_model('DenseNet121_apple_final.h5')
model_eff = load_model('EfficientNetB0_apple_final.h5')

In [13]:
print("Predicting with ResNet50...")
pred_res = model_res.predict(test_gen, verbose=1)

print("Predicting with DenseNet121...")
pred_dense = model_dense.predict(test_gen, verbose=1)

print("Predicting with EfficientNetB0...")
pred_eff = model_eff.predict(test_gen, verbose=1)

Predicting with ResNet50...
4/4 ━━━━━━━━━━━━━━━━━━━━ 16s 4s/step
Predicting with DenseNet121...
4/4 ━━━━━━━━━━━━━━━━━━━━ 19s 4s/step
Predicting with EfficientNetB0...


3/4 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/stepWARNING:tensorflow:6 out of the last 12 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x7adbde19b920> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.


4/4 ━━━━━━━━━━━━━━━━━━━━ 11s 2s/step


In [14]:
ensemble_preds = (pred_res + pred_dense + pred_eff) / 3.0
final_classes = np.argmax(ensemble_preds, axis=1)

In [15]:
true_classes = test_gen.classes
class_labels = list(test_gen.class_indices.keys())

In [16]:
acc = accuracy_score(true_classes, final_classes)
print(f"\n🏆 ENSEMBLE ACCURACY: {acc*100:.2f}%")


🏆 ENSEMBLE ACCURACY: 73.33%
